In [81]:
import pandas as pd
import numpy as np

In [82]:
def removeWav(fileName):
    return fileName.split('_')[0]

#Spanish

In [83]:
def cleanupIdInAcoustic(id):
  return id.split('.')[0].split('trimmed_')[1].split('_')[0]

##Acoustic

In [84]:
spaAcousticDF = pd.read_excel('SPA_speechTimingMeasures.xlsx')
spaAcousticDF['Unnamed: 0'] = spaAcousticDF['Unnamed: 0'].apply(cleanupIdInAcoustic)
#rename 'Unnamed: 0' to 'Sample_ID'
spaAcousticDF = spaAcousticDF.rename(columns={'Unnamed: 0': 'Sample_ID'})
spaAcousticDF.head()

,Sample_ID,Mean Pause Duration,Variability of Pause Duration,# of syllables,speechrate(nsyll / dur),Average Syllable Duration (speakingtot/voicedcount),articulation rate(nsyll / phonationtime),Speech-to-pause ratio,Time (secs),# of pauses
0,BILP006,1.213500,1.288158,702,2.188507,0.235668,4.243264,1.065092,320.766667,129
1,BILP007,1.278933,0.711372,229,2.396601,0.249712,4.004617,1.490409,95.552000,31
2,BILP008,0.703200,0.092403,309,4.005635,0.204134,4.898749,4.485021,77.141333,21
3,BILP009,0.767030,0.247311,386,3.375352,0.230690,4.334817,3.517953,114.358437,34
4,BILP010,1.568102,1.750063,270,1.809513,0.227399,4.397558,0.699181,149.211429,56


##Linguistic

In [85]:
def cleanupIdInLing(id):
  return id.split('_')[0]

In [86]:
propositionalDensityPOSTags = ['VERB','ADJ','ADP','ADV','CCONJ','SCONJ']
propositionalDensityPOSTags = ['POS_COUNT:' + tag for tag in propositionalDensityPOSTags]

In [87]:
spaDominantLingDF = pd.read_csv('SPA_Dom_LinguisticFeatures.csv')
spaDominantLingDF['# Propositions'] = spaDominantLingDF[propositionalDensityPOSTags].sum(axis=1)
spaDominantLingDF = spaDominantLingDF[['file','# of words','# Propositions']]
spaDominantLingDF['file'] = spaDominantLingDF['file'].apply(cleanupIdInLing)
spaDominantLingDF = spaDominantLingDF.rename(columns={'file': 'Sample_ID'})
spaDominantLingDF.head()

,Sample_ID,# of words,# Propositions
0,BILP006,402.0,189.0
1,BILP009,128.0,57.0
2,BILP013,301.0,128.0
3,BILP022,141.0,64.0
4,BILP024,172.0,75.0


In [88]:
spaNonDominantLingDF = pd.read_csv('SPA_NotDom_LinguisticFeatures.csv')
spaNonDominantLingDF['# Propositions'] = spaNonDominantLingDF[propositionalDensityPOSTags].sum(axis=1)
spaNonDominantLingDF = spaNonDominantLingDF[['file','# of words','# Propositions']]
spaNonDominantLingDF['file'] = spaNonDominantLingDF['file'].apply(cleanupIdInLing)
spaNonDominantLingDF = spaNonDominantLingDF.rename(columns={'file': 'Sample_ID'})
spaNonDominantLingDF.head()

,Sample_ID,# of words,# Propositions
0,BILP007,84.0,35.0
1,BILP008,181.0,77.0
2,BILP010,97.0,38.0
3,BILP011,87.0,33.0
4,BILP012,97.0,30.0


In [89]:
spaLingDF = pd.concat([spaDominantLingDF, spaNonDominantLingDF])
spaLingDF = spaLingDF.sort_values(by='Sample_ID')
spaLingDF.head()

,Sample_ID,# of words,# Propositions
0,BILP006,402.0,189.0
0,BILP007,84.0,35.0
1,BILP008,181.0,77.0
1,BILP009,128.0,57.0
2,BILP010,97.0,38.0


In [90]:
assert list(spaLingDF['Sample_ID']) == list(spaAcousticDF['Sample_ID']), print('Sample IDs do not match')

In [91]:
spaDF = pd.merge(spaAcousticDF, spaLingDF)
spaDF.head()

,Sample_ID,Mean Pause Duration,Variability of Pause Duration,# of syllables,speechrate(nsyll / dur),Average Syllable Duration (speakingtot/voicedcount),articulation rate(nsyll / phonationtime),Speech-to-pause ratio,Time (secs),# of pauses,# of words,# Propositions
0,BILP006,1.213500,1.288158,702,2.188507,0.235668,4.243264,1.065092,320.766667,129,402.0,189.0
1,BILP007,1.278933,0.711372,229,2.396601,0.249712,4.004617,1.490409,95.552000,31,84.0,35.0
2,BILP008,0.703200,0.092403,309,4.005635,0.204134,4.898749,4.485021,77.141333,21,181.0,77.0
3,BILP009,0.767030,0.247311,386,3.375352,0.230690,4.334817,3.517953,114.358437,34,128.0,57.0
4,BILP010,1.568102,1.750063,270,1.809513,0.227399,4.397558,0.699181,149.211429,56,97.0,38.0


In [92]:
spaDF.columns

Index(['Sample_ID', 'Mean Pause Duration', 'Variability of Pause Duration',
       '# of syllables', 'speechrate(nsyll / dur)',
       'Average Syllable Duration (speakingtot/voicedcount)',
       'articulation rate(nsyll / phonationtime)', 'Speech-to-pause ratio',
       'Time (secs)', '# of pauses', '# of words', '# Propositions'],
      dtype='object')

In [93]:
spaDF['# words/min'] = spaDF['# of words']/(spaDF['Time (secs)']/60)
spaDF['# propositions/min'] = spaDF['# Propositions']/(spaDF['Time (secs)']/60)
spaDF = spaDF.drop(columns=['# of words','Time (secs)', '# of syllables','# Propositions'])
spaAcousticDF = spaDF.rename(columns={'Average Syllable Duration (speakingtot/voicedcount)': 'Average Syllable Duration', 'articulation rate(nsyll / phonationtime)': 'Articulation rate', 'speechrate(nsyll / dur)':'speechrate'})
spaAcousticDF.head()

,Sample_ID,Mean Pause Duration,Variability of Pause Duration,speechrate,Average Syllable Duration,Articulation rate,Speech-to-pause ratio,# of pauses,# words/min,# propositions/min
0,BILP006,1.213500,1.288158,2.188507,0.235668,4.243264,1.065092,129,75.194846,35.352801
1,BILP007,1.278933,0.711372,2.396601,0.249712,4.004617,1.490409,31,52.746149,21.977562
2,BILP008,0.703200,0.092403,4.005635,0.204134,4.898749,4.485021,21,140.780559,59.890072
3,BILP009,0.767030,0.247311,3.375352,0.230690,4.334817,3.517953,34,67.157266,29.905970
4,BILP010,1.568102,1.750063,1.809513,0.227399,4.397558,0.699181,56,39.005055,15.280331


In [94]:
spaAcousticDF.columns, spaAcousticDF.shape

(Index(['Sample_ID', 'Mean Pause Duration', 'Variability of Pause Duration',
        'speechrate', 'Average Syllable Duration', 'Articulation rate',
        'Speech-to-pause ratio', '# of pauses', '# words/min',
        '# propositions/min'],
       dtype='object'),
 (34, 10))

In [95]:
spaAcousticDF.to_csv('spaAcoustic_Preprocessed_May1.csv', index=False)
from google.colab import files
files.download('spaAcoustic_Preprocessed_May1.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Catalan

In [96]:
def cleanupIdInAcoustic(id):
  return id.split('.')[0].split('trimmed_')[1].split('_')[0]

##Acoustic

In [120]:
catAcousticDF = pd.read_excel('CAT_speechTimingMeasures_May1.xlsx')
catAcousticDF['Sample_ID'] = catAcousticDF['Sample_ID'].apply(cleanupIdInAcoustic)
catAcousticDF.head()

,Sample_ID,# of syllables,speechrate(nsyll / dur),Average Syllable Duration (speakingtot/voicedcount),articulation rate(nsyll / phonationtime),Speech-to-pause ratio,Time (secs),# of pauses,Mean Pause Duration,Variability of Pause Duration
0,BILP006,526,2.249347,0.222581,4.492747,1.002651,233.845594,99,1.191510,1.127368
1,BILP007,238,2.035069,0.226106,4.422696,0.852340,116.949333,38,1.706378,2.344119
2,BILP008,256,3.911343,0.198854,5.028811,3.500183,65.450667,21,0.727200,0.156940
3,BILP009,356,3.083426,0.247910,4.033720,3.244706,115.456000,38,0.735135,0.192954
4,BILP010,423,2.597004,0.195858,5.105736,1.035186,162.880000,68,1.194507,0.735932


##Linguistic

In [121]:
def cleanupIdInLing(id):
  return id.split('_')[0]

In [122]:
propositionalDensityPOSTags = ['VERB','ADJ','ADP','ADV','CCONJ','SCONJ']
propositionalDensityPOSTags = ['POS_COUNT:' + tag for tag in propositionalDensityPOSTags]

In [123]:
catDominantLingDF = pd.read_csv('CAT_Dom_LinguisticFeatures.csv')
catDominantLingDF['# Propositions'] = catDominantLingDF[propositionalDensityPOSTags].sum(axis=1)
catDominantLingDF = catDominantLingDF[['file','# of words','# Propositions']]
catDominantLingDF['file'] = catDominantLingDF['file'].apply(cleanupIdInLing)
catDominantLingDF = catDominantLingDF.rename(columns={'file': 'Sample_ID'})
catDominantLingDF.head()

,Sample_ID,# of words,# Propositions
0,BILP007,110.0,52.0
1,BILP008,184.0,86.0
2,BILP010,172.0,92.0
3,BILP011,77.0,25.0
4,BILP012,103.0,41.0


In [124]:
catNonDominantLingDF = pd.read_csv('CAT_NotDom_LinguisticFeatures.csv')
catNonDominantLingDF['# Propositions'] = catNonDominantLingDF[propositionalDensityPOSTags].sum(axis=1)
catNonDominantLingDF = catNonDominantLingDF[['file','# of words','# Propositions']]
catNonDominantLingDF['file'] = catNonDominantLingDF['file'].apply(cleanupIdInLing)
catNonDominantLingDF = catNonDominantLingDF.rename(columns={'file': 'Sample_ID'})
catNonDominantLingDF.head()

,Sample_ID,# of words,# Propositions
0,BILP006,206.0,86.0
1,BILP009,76.0,32.0
2,BILP013,260.0,114.0
3,BILP022,266.0,125.0
4,BILP024,180.0,78.0


In [125]:
catLingDF = pd.concat([catDominantLingDF, catNonDominantLingDF])
catLingDF = catLingDF.sort_values(by='Sample_ID')
catLingDF.head()

,Sample_ID,# of words,# Propositions
0,BILP006,206.0,86.0
0,BILP007,110.0,52.0
1,BILP008,184.0,86.0
1,BILP009,76.0,32.0
2,BILP010,172.0,92.0


In [126]:
 print(list(catLingDF['Sample_ID']))
 print(list(catAcousticDF['Sample_ID']))

['BILP006', 'BILP007', 'BILP008', 'BILP009', 'BILP010', 'BILP011', 'BILP012', 'BILP013', 'BILP014', 'BILP015', 'BILP017', 'BILP018', 'BILP019', 'BILP020', 'BILP021', 'BILP022', 'BILP024', 'BILP025', 'BILP026', 'BILP027', 'BILP028', 'BILP029', 'BIOBS003', 'BIOBS004', 'BIOBS005', 'BISE004', 'BISE005', 'BISE010', 'BISE011', 'BISE013', 'BISE014', 'BISE016', 'BISE018', 'BISE019']
['BILP006', 'BILP007', 'BILP008', 'BILP009', 'BILP010', 'BILP011', 'BILP012', 'BILP013', 'BILP014', 'BILP015', 'BILP017', 'BILP018', 'BILP019', 'BILP020', 'BILP021', 'BILP022', 'BILP024', 'BILP025', 'BILP026', 'BILP027', 'BILP028', 'BILP029', 'BIOBS003', 'BIOBS004', 'BIOBS005', 'BISE004', 'BISE005', 'BISE010', 'BISE011', 'BISE013', 'BISE014', 'BISE016', 'BISE018', 'BISE019']


In [127]:
len(list(catLingDF['Sample_ID'])), len(list(catAcousticDF['Sample_ID']))

(34, 34)

In [128]:
assert list(catLingDF['Sample_ID']) == list(catAcousticDF['Sample_ID']), print('Sample IDs do not match')

In [131]:
catDF = pd.merge(catAcousticDF, catLingDF)
catDF.head()

,Sample_ID,# of syllables,speechrate(nsyll / dur),Average Syllable Duration (speakingtot/voicedcount),articulation rate(nsyll / phonationtime),Speech-to-pause ratio,Time (secs),# of pauses,Mean Pause Duration,Variability of Pause Duration,# of words,# Propositions
0,BILP006,526,2.249347,0.222581,4.492747,1.002651,233.845594,99,1.191510,1.127368,206.0,86.0
1,BILP007,238,2.035069,0.226106,4.422696,0.852340,116.949333,38,1.706378,2.344119,110.0,52.0
2,BILP008,256,3.911343,0.198854,5.028811,3.500183,65.450667,21,0.727200,0.156940,184.0,86.0
3,BILP009,356,3.083426,0.247910,4.033720,3.244706,115.456000,38,0.735135,0.192954,76.0,32.0
4,BILP010,423,2.597004,0.195858,5.105736,1.035186,162.880000,68,1.194507,0.735932,172.0,92.0


In [130]:
catDF.columns

Index(['Sample_ID', 'Mean Pause Duration', 'Variability of Pause Duration',
       'speechrate', 'Average Syllable Duration', 'Articulation rate',
       'Speech-to-pause ratio', '# of pauses', '# words/min',
       '# propositions/min', '# of words', '# Propositions'],
      dtype='object')

In [132]:
catDF['# words/min'] = catDF['# of words']/(catDF['Time (secs)']/60)
catDF['# propositions/min'] = catDF['# Propositions']/(catDF['Time (secs)']/60)
catDF = catDF.drop(columns=['# of words','Time (secs)', '# of syllables','# Propositions'])
catAcousticDF = catDF.rename(columns={'Average Syllable Duration (speakingtot/voicedcount)': 'Average Syllable Duration', 'articulation rate(nsyll / phonationtime)': 'Articulation rate', 'speechrate(nsyll / dur)':'speechrate'})
catAcousticDF.head()

,Sample_ID,speechrate,Average Syllable Duration,Articulation rate,Speech-to-pause ratio,# of pauses,Mean Pause Duration,Variability of Pause Duration,# words/min,# propositions/min
0,BILP006,2.249347,0.222581,4.492747,1.002651,99,1.191510,1.127368,52.855390,22.065842
1,BILP007,2.035069,0.226106,4.422696,0.852340,38,1.706378,2.344119,56.434695,26.678220
2,BILP008,3.911343,0.198854,5.028811,3.500183,21,0.727200,0.156940,168.676662,78.838005
3,BILP009,3.083426,0.247910,4.033720,3.244706,38,0.735135,0.192954,39.495565,16.629712
4,BILP010,2.597004,0.195858,5.105736,1.035186,68,1.194507,0.735932,63.359528,33.889980


In [133]:
catAcousticDF.columns, catAcousticDF.shape

(Index(['Sample_ID', 'speechrate', 'Average Syllable Duration',
        'Articulation rate', 'Speech-to-pause ratio', '# of pauses',
        'Mean Pause Duration', 'Variability of Pause Duration', '# words/min',
        '# propositions/min'],
       dtype='object'),
 (34, 10))

In [134]:
catAcousticDF.to_csv('catAcoustic_Preprocessed_May1.csv', index=False)
from google.colab import files
files.download('catAcoustic_Preprocessed_May1.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>